In [1]:
print("olá")

olá


In [11]:
import os
from dotenv import load_dotenv
load_dotenv()

import re
import ast
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_community.tools import YouTubeSearchTool
from langchain_community.document_loaders import YoutubeLoader


import re
from urllib.parse import urlparse, parse_qs

def extract_video_ids(urls: list[str]) -> list[str]:
    video_ids = []
    for url in urls:
        parsed = urlparse(url)

        if parsed.hostname in ("www.youtube.com", "youtube.com"):
            qs = parse_qs(parsed.query)
            if "v" in qs:
                video_ids.append(qs["v"][0])
                continue
            match = re.match(r"^/embed/([a-zA-Z0-9_-]{11})", parsed.path)
            if match:
                video_ids.append(match.group(1))
                continue

        if parsed.hostname in ("youtu.be",):
            match = re.match(r"^/([a-zA-Z0-9_-]{11})", parsed.path)
            if match:
                video_ids.append(match.group(1))
                continue

        match = re.search(r"(?:v=|/)([a-zA-Z0-9_-]{11})", url)
        if match:
            video_ids.append(match.group(1))

    return video_ids

def extract_video_ids_from_pattern(urls) -> list[str]:
    """Extrai IDs de vídeos usando pattern de URL"""
    pattern = r'v=([^"]+?)&pp='
    return [re.search(pattern, url).group(1) for url in urls if re.search(pattern, url)]


def get_youtube_urls(video_query: str, number_of_videos: int=1, language: str="pt"):
    """
    Busca vídeos no YouTube com suporte a idioma
    
    Args:
        video_query: Termo de busca
        number_of_videos: Número de vídeos a retornar
        language: Código do idioma (pt, pt-BR, en, es, etc.)
    
    Returns:
        Lista com URLs dos vídeos encontrados
    """
    # Mapear idiomas para região/host_language
    language_map = {
        "pt": "pt",      # Portugal
        "pt-BR": "pt",   # Brasil (usa Portuguese)
        "en": "en",
        "es": "es",
    }
    
    lang_code = language_map.get(language, "pt")
    
    try:
        # Adiciona parâmetro de idioma na busca
        search_query = f"{video_query},{number_of_videos}"
        results = ast.literal_eval(YouTubeSearchTool().run(search_query))
        return results
    except Exception as e:
        print(f"⚠️ Erro na busca: {e}")
        return []

# Teste com português
video_ids = extract_video_ids(get_youtube_urls("qualidade pior no gpt5", 5, language="pt-BR"))

print(f"Vídeos encontrados: {len(video_ids)}")
for vid_id in video_ids:
    print(f"  - {vid_id}")

Vídeos encontrados: 5
  - dlGyZcsLYus
  - ZUTe_lxJzvI
  - wSZfaKrTEdg
  - QQYZ99SPGDU
  - dVNRzD0woCY


In [ ]:
print("🔍 Buscando vídeos em português...")
urls = get_youtube_urls("inteligência artificial", number_of_videos=3, language="pt-BR")

print("\n📺 Extraindo IDs dos vídeos...")
video_ids = extract_video_ids(urls)
print(f"Total de vídeos encontrados: {len(video_ids)}")

print("\n📝 Obtendo transcrições...")
for idx, vid_id in enumerate(video_ids[:1], 1):
    print(f"\n[Vídeo {idx}] ID: {vid_id}")
    
    try:
        transcript = YouTubeTranscriptApi().fetch(
            video_id=vid_id,
            languages=["pt", "pt-BR", "en"]
        )
        
        full_text = " ".join([seg.text for seg in transcript.snippets])
        print(f"✅ Transcrição obtida: {len(full_text)} caracteres")
        print(f"\nPrimeiros 300 caracteres:\n{full_text[:300]}...")
        
    except Exception as e:
        print(f"❌ Erro: {e}")

🔍 Buscando vídeos em português...

📺 Extraindo IDs dos vídeos...
Total de vídeos encontrados: 3

📝 Obtendo transcrições...

[Vídeo 1] ID: 6KUTRsJNTno
✅ Transcrição obtida: 2203 caracteres

Primeiros 300 caracteres:
Ainda falando sobre essa guerra, os Estados Unidos contaram com o suporte de inteligência artificial no ataque que fizeram ao Irã, seja mapeando alvos ou coletando informações. As ferramentas de IA se mostraram tão fundamentais que acabaram mudando o xadrez desse conflito. Quando os Estados Unidos l...


In [23]:
full_text = " ".join([seg.text for seg in transcript.snippets])

In [24]:
full_text

'Ainda falando sobre essa guerra, os Estados Unidos contaram com o suporte de inteligência artificial no ataque que fizeram ao Irã, seja mapeando alvos ou coletando informações. As ferramentas de IA se mostraram tão fundamentais que acabaram mudando o xadrez desse conflito. Quando os Estados Unidos lançaram o ataque [música] aéreo contra o Irã, começava também a primeira Grande Guerra com o uso massivo de inteligência artificial no campo de batalha. >> Funciona assim. Grandes decisões dos grandes responsáveis pela guerra agora são tomadas baseadas num outro chefão, o algoritmo. Pelo ar e pelo mar, a Iá vai processando uma quantidade impressionante de dados que chegam por imagens e sensores, facilitando o mapeamento dos alvos que eles querem atingir. Além disso, o uso de drones camicazes guiados foi facilitado pelo auxílio da ferramenta, mas ela ajuda também a definir estratégias. O especialista em negócios internacionais, Benfard diz que a IA tem capacidade de redefinir o xadrez geopol

In [1]:
from youtube_transcript_collector import YouTubeTranscriptCollector

# ===== EXEMPLO DE USO DA CLASSE =====

# Criar uma instância do collector
collector = YouTubeTranscriptCollector(
    max_videos=3,
    languages=["pt", "pt-BR", "en"],
    verbose=True
)

# Executar a coleta completa com uma única chamada
results = collector.collect("machine learning em português")

print("\n" + "="*60)
print("RESULTADOS:")
print("="*60)

# Acessar os resultados
for idx, transcript in enumerate(results, 1):
    print(f"\n[Vídeo {idx}]")
    print(f"  ID: {transcript.video_id}")
    print(f"  Idioma: {transcript.language}")
    print(f"  Tamanho: {len(transcript.transcript_text)} caracteres")
    print(f"  Primeiros 200 caracteres:")
    print(f"  {transcript.transcript_text[:200]}...")

# Obter um resumo dos resultados
summary = collector.get_results_summary()
print(f"\n\n📊 RESUMO: {summary['total_transcripts']} transcrição(ões) coletada(s)")

# Obter todo o texto combinado
full_text = collector.get_combined_text()
print(f"📄 Texto combinado: {len(full_text)} caracteres no total")


YouTube Transcript Collection Started
🔍 Searching for 'machine learning em português' (max 3 videos)...
✅ Found 3 URLs

📺 Extracting video IDs...
✅ Extracted 3 video IDs

📝 Retrieving transcripts...

[1/3] Processing video...
  📥 Fetching transcript for Iuz_jc96bQk...
  ❌ Failed to get transcript: 'FetchedTranscriptSnippet' object is not subscriptable

[2/3] Processing video...
  📥 Fetching transcript for 0mtXae5HhTE...
  ❌ Failed to get transcript: 'FetchedTranscriptSnippet' object is not subscriptable

[3/3] Processing video...
  📥 Fetching transcript for 0PrOA2JK6GQ...
  ❌ Failed to get transcript: 'FetchedTranscriptSnippet' object is not subscriptable

Collection Complete: 0 transcripts retrieved


RESULTADOS:


📊 RESUMO: 0 transcrição(ões) coletada(s)
📄 Texto combinado: 0 caracteres no total
